# 04 · Business Analysis — KPIs, Business Questions & Recommendations

EDA (notebook 03) told us **what** is happening. This notebook turns it into answers management can act on:

1. Define the **KPIs** precisely.
2. Answer the **business questions** one by one — every finding below is *computed from the data in this notebook*, not typed in by hand.
3. Investigate the two biggest issues in depth: **2024 growth without profit** and **Central's low margin**.
4. Size the opportunity with a clearly-labelled **what-if** scenario.
5. Produce **recommendations**, each linked to evidence and to a KPI that can track it.

Language used throughout:
* **Fact** — directly measured in the data.
* **Hypothesis** — a possible explanation that fits the facts but is not proven.
* **Recommendation** — a proposed action, with how to verify it worked.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

import eda
from feature_engineering import aggregate_margin
from utils import CLEAN_FILE, money

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 110)

df = pd.read_csv(CLEAN_FILE, parse_dates=["Order Date", "Ship Date"])
LATEST = int(df["Year"].max())

---
## 1 · KPI definitions

First, the words people often mix up:

| Term | Meaning | In this data |
|---|---|---|
| **Sales / Revenue** | Money received from customers, *after* discount. (In this project "Sales" and "Revenue" mean the same thing.) | `SUM(Sales)` |
| **Gross Sales** | What the same items would have earned at full list price | `SUM(Sales / (1 − Discount))` |
| **Profit** | Money left after the cost of the goods: Sales − Cost | `SUM(Profit)` |
| **Profit Margin** | Profit as a % of sales — *efficiency*, not size | `SUM(Profit) / SUM(Sales)` |
| **Order count** | Number of separate customer orders | `COUNT DISTINCT Order ID` |
| **Order lines** | Number of rows (product lines). One order can have many | `COUNT rows` |
| **Quantity** | Number of physical units sold | `SUM(Quantity)` |

| KPI | Formula | Business meaning | Why it matters | Limitation |
|---|---|---|---|---|
| Total Sales | SUM(Sales) | Size of the business | Top-line growth | Says nothing about profitability |
| Total Profit | SUM(Profit) | Money actually earned | What pays the bills | Bigger regions always look better |
| Profit Margin % | SUM(Profit)/SUM(Sales) | Profit per $1 sold | Fair comparison across sizes | A high % on tiny sales can be unimportant |
| Total Orders | DISTINCT Order ID | Customer purchase events | Demand / activity | Treats a $5 and a $5,000 order the same |
| Total Quantity | SUM(Quantity) | Units shipped | Logistics & inventory | Adds pens and copiers together — mixes very different items |
| Average Order Value | SUM(Sales)/Orders | Spend per order | Basket size | Skewed by a few huge orders |
| YoY Sales Growth % | (This year − last year)/last year | Momentum | Is the business growing? | Sensitive to one unusual year; compare full years only |
| YoY Profit Growth % | same, on Profit | Profit momentum | Detects growth without profit | Unstable when last year's profit is near zero |
| Effective Discount % | Discount given / Gross Sales | Price given away | Leading indicator of margin erosion | Doesn't show *why* the discount was given |
| Top Region / Category | Highest Sales | Where revenue concentrates | Focus & resourcing | "Top" by sales may not be top by profit |

In [2]:
def kpis(frame):
    orders = frame["Order ID"].nunique()
    return pd.Series({
        "Total Sales": frame["Sales"].sum(),
        "Total Profit": frame["Profit"].sum(),
        "Profit Margin %": aggregate_margin(frame["Sales"], frame["Profit"]),
        "Total Orders": orders,
        "Total Quantity": frame["Quantity"].sum(),
        "Average Order Value": frame["Sales"].sum() / orders,
        "Effective Discount %": frame["Discount Amount"].sum() / frame["Gross Sales"].sum() * 100,
    })

kpi_table = pd.DataFrame({"All years": kpis(df),
                          str(LATEST - 1): kpis(df[df["Year"] == LATEST - 1]),
                          str(LATEST): kpis(df[df["Year"] == LATEST])})
# Amounts change in %, ratios change in percentage POINTS (pp): 10.6% -> 9.2% is -1.3 pp
ratio_rows = ["Profit Margin %", "Effective Discount %"]
change = (kpi_table[str(LATEST)] / kpi_table[str(LATEST - 1)] - 1) * 100
change[ratio_rows] = kpi_table.loc[ratio_rows, str(LATEST)] - kpi_table.loc[ratio_rows, str(LATEST - 1)]
kpi_table["YoY change (% or pp)"] = change
kpi_table

,All years,2023,2024,YoY change (% or pp)
Total Sales,"4,391,176.00","1,117,829.65","1,192,590.88",6.69
Total Profit,"452,068.04","118,071.19","110,241.76",-6.63
Profit Margin %,10.29,10.56,9.24,-1.32
Total Orders,"5,190.00","1,353.00","1,393.00",2.96
Total Quantity,"56,703.00","14,611.00","15,341.00",5.00
Average Order Value,846.08,826.19,856.13,3.62
Effective Discount %,15.50,15.59,16.17,0.58


In [3]:
region_s = eda.summarize_by(df, "Region")
cat_s = eda.summarize_by(df, "Category")
print("Top region by sales  :", region_s["Sales"].idxmax())
print("Top region by profit :", region_s["Profit"].idxmax())
print("Top region by margin :", region_s["Profit Margin %"].idxmax())
print("Top category by sales:", cat_s["Sales"].idxmax())
print("Top category by margin:", cat_s["Profit Margin %"].idxmax())

Top region by sales  : West
Top region by profit : West
Top region by margin : East
Top category by sales: Technology
Top category by margin: Office Supplies


---
## 2 · Business questions

Each answer is built from the data below. Columns: **Question · Required columns · Calculation · Visualization · Finding · Business implication**.

In [4]:
reg = eda.summarize_by(df, "Region")
cat = eda.summarize_by(df, "Category")
sub = eda.summarize_by(df, "Sub-Category")
seg = eda.summarize_by(df, "Segment")
yr = df.groupby("Year")[["Sales", "Profit"]].sum()
reg_cagr = eda.yearly_pivot(df, "Region", "Sales").iloc[:, -1]
hslp = eda.high_sales_low_profit(df)
disc = eda.discount_impact(df)
corr = eda.discount_correlation(df)
cust = df.groupby("Customer Name")["Sales"].sum().sort_values(ascending=False)
cust_id = df.groupby("Customer ID")["Sales"].sum().sort_values(ascending=False)
season = eda.seasonality(df)["Avg Monthly Sales"]
month_name = lambda m: pd.Timestamp(2000, m, 1).strftime("%b")
loss_subs = sub[sub["Profit"] < 0]

def pct(x):
    return f"{x:.1f}%"

questions = [
 ("Which region has the highest sales?", "Region, Sales", "SUM(Sales) by Region", "sales_by_region.png",
  f"{reg['Sales'].idxmax()} — {money(reg['Sales'].max())} ({pct(reg['Sales Share %'].max())} of total)",
  "Core market; protect its growth."),
 ("Which region has the highest profit?", "Region, Profit", "SUM(Profit) by Region", "profit_by_region.png",
  f"{reg['Profit'].idxmax()} — {money(reg['Profit'].max())}; lowest is {reg['Profit'].idxmin()} at {money(reg['Profit'].min())}",
  "Central ranks 3rd in sales but last in profit."),
 ("Which region has the highest profit margin?", "Region, Sales, Profit", "SUM(Profit)/SUM(Sales) by Region", "regional_profit_margin.png",
  f"{reg['Profit Margin %'].idxmax()} {pct(reg['Profit Margin %'].max())}; lowest {reg['Profit Margin %'].idxmin()} {pct(reg['Profit Margin %'].min())} (company {pct(aggregate_margin(df['Sales'], df['Profit']))})",
  "Central is the main drag on company margin."),
 ("Which category generates the most revenue?", "Category, Sales", "SUM(Sales) by Category", "category_analysis.png",
  f"{cat['Sales'].idxmax()} — {money(cat['Sales'].max())} ({pct(cat['Sales Share %'].max())})",
  "Technology availability and pricing matter most for revenue."),
 ("Which category generates the most profit?", "Category, Profit", "SUM(Profit) by Category", "category_analysis.png",
  f"{cat['Profit'].idxmax()} — {money(cat['Profit'].max())}; best margin: {cat['Profit Margin %'].idxmax()} {pct(cat['Profit Margin %'].max())}; Furniture margin {pct(cat.loc['Furniture', 'Profit Margin %'])}",
  "Furniture's $1.45M of sales earns almost nothing."),
 ("Which products have high sales but low profit?", "Product Name, Sales, Profit", "Top 25% by sales AND margin < 5%", "product_sales_vs_profit.png",
  f"{len(hslp)} products; {(hslp.index.str.contains('Table|Bookcase|Shelving')).sum()} are tables/bookcases; combined profit {money(hslp['Profit'].sum())}",
  "They look successful on sales reports but destroy profit."),
 ("Which customers generate the most revenue?", "Customer, Sales", "SUM(Sales) by Customer, top 10; share of top 20%", "top_customers.png",
  f"Top: {cust.index[0]} ({money(cust.iloc[0])}). Top 20% of customers = {pct(cust_id.head(int(len(cust_id)*0.2)).sum()/cust_id.sum()*100)} of sales",
  "Retention of the top ~150 customers is a priority."),
 ("How does discount affect profitability?", "Discount, Sales, Profit", "Margin per discount level; correlation", "discount_vs_margin.png",
  f"Margin {pct(disc.loc[0.0, 'Profit Margin %'])} at 0% → {pct(disc.loc[0.2, 'Profit Margin %'])} at 20%; every level ≥30% loses money. Pearson r = {corr['Pearson']}",
  "Strong negative association; 30%+ discounts need control."),
 ("Which months have the highest sales?", "Order Date, Sales", "Average monthly sales by calendar month", "seasonality.png",
  "Top: " + ", ".join(month_name(m) for m in season.nlargest(3).index) + "; weakest: " + ", ".join(month_name(m) for m in season.nsmallest(2).index),
  "Plan stock, staff and campaigns for Sep–Dec."),
 ("Is sales growth improving?", "Year, Sales, Profit", "YoY % on yearly totals", "monthly_sales.png",
  f"Sales YoY: {pct((yr['Sales'].pct_change()*100).iloc[-2])} ({LATEST-1}), {pct((yr['Sales'].pct_change()*100).iloc[-1])} ({LATEST}). Profit YoY {LATEST}: {pct((yr['Profit'].pct_change()*100).iloc[-1])}",
  "Sales growth is real, but 2024 profit fell → growth without profit."),
 ("Which region has declining performance?", "Region, Year, Sales", "CAGR of yearly sales by region", "yearly_sales_by_region.png",
  ", ".join(f"{r} {pct(v)}" for r, v in reg_cagr.sort_values().items()) + " per year",
  "South is shrinking; Central is flat-to-declining with collapsing profit."),
 ("Which sub-category is most profitable?", "Sub-Category, Profit", "SUM(Profit) by Sub-Category", "subcategory_profit.png",
  f"By profit: {sub['Profit'].idxmax()} ({money(sub['Profit'].max())}); by margin: {sub['Profit Margin %'].idxmax()} ({pct(sub['Profit Margin %'].max())})",
  "Protect Phones/Chairs/Copiers; Paper is small but very efficient."),
 ("Which sub-category is least profitable?", "Sub-Category, Profit", "SUM(Profit) by Sub-Category", "subcategory_profit.png",
  "; ".join(f"{s} {money(p)}" for s, p in loss_subs['Profit'].sort_values().items()),
  "Tables and Bookcases need a pricing/cost review."),
 ("Which customer segment generates the most sales?", "Segment, Sales, Profit", "SUM(Sales), margin by Segment", "segment_performance.png",
  f"{seg['Sales'].idxmax()} {pct(seg['Sales Share %'].max())} of sales; best margin {seg['Profit Margin %'].idxmax()} {pct(seg['Profit Margin %'].max())}",
  "Segments differ little; not the main profit lever."),
 ("Where should management focus improvement efforts?", "Region, Sub-Category, Discount, Profit", "Profit matrix Region × Sub-Category; loss by discount band", "region_subcategory_profit_heatmap.png",
  f"Net profit on lines with ≥30% discount: {money(df.loc[df['Discount'] >= 0.3, 'Profit'].sum())}; Tables + Bookcases: {money(sub.loc[['Tables','Bookcases'], 'Profit'].sum())}",
  "1) discount control (esp. Central), 2) Tables/Bookcases, 3) South decline."),
 ("Did profit grow with sales in the latest year?", "Year, Sales, Profit, Discount", "YoY Sales vs YoY Profit; effective discount by year", "(section 3 below)",
  f"No: sales {pct((yr['Sales'].pct_change()*100).iloc[-1])}, profit {pct((yr['Profit'].pct_change()*100).iloc[-1])}",
  "A sales-only target would have rewarded value-destroying growth."),
 ("Is Central's low margin caused by product mix or discounting?", "Region, Sub-Category, Discount, Profit", "Effective discount by Region × Sub-Category; margin on <30% lines", "region_subcategory_profit_heatmap.png",
  "Central discounts most in 15 of 17 sub-categories; on <30%-discount lines its margin matches other regions",
  "Points to discount practice, not product mix (association, not proof)."),
]
bq = pd.DataFrame(questions, columns=["Question", "Required columns", "Calculation", "Visualization", "Finding", "Business implication"])
bq.index = range(1, len(bq) + 1)
bq

,Question,Required columns,Calculation,Visualization,Finding,Business implication
1,Which region has the highest sales?,"Region, Sales",SUM(Sales) by Region,sales_by_region.png,West — $1.51M (34.5% of total),Core market; protect its growth.
2,Which region has the highest profit?,"Region, Profit",SUM(Profit) by Region,profit_by_region.png,West — $186K; lowest is Central at $29K,Central ranks 3rd in sales but last in profit.
3,Which region has the highest profit margin?,"Region, Sales, Profit",SUM(Profit)/SUM(Sales) by Region,regional_profit_margin.png,East 13.2%; lowest Central 3.3% (company 10.3%),Central is the main drag on company margin.
4,Which category generates the most revenue?,"Category, Sales",SUM(Sales) by Category,category_analysis.png,Technology — $1.82M (41.5%),Technology availability and pricing matter most for revenue.
5,Which category generates the most profit?,"Category, Profit",SUM(Profit) by Category,category_analysis.png,Technology — $253K; best margin: Office Supplies 18.0%; Furniture margin -0.1%,Furniture's $1.45M of sales earns almost nothing.
6,Which products have high sales but low profit?,"Product Name, Sales, Profit",Top 25% by sales AND margin < 5%,product_sales_vs_profit.png,8 products; 7 are tables/bookcases; combined profit -$97K,They look successful on sales reports but destroy profit.
7,Which customers generate the most revenue?,"Customer, Sales","SUM(Sales) by Customer, top 10; share of top 20%",top_customers.png,Top: Grace Diaz ($70K). Top 20% of customers = 54.3% of sales,Retention of the top ~150 customers is a priority.
8,How does discount affect profitability?,"Discount, Sales, Profit",Margin per discount level; correlation,discount_vs_margin.png,Margin 24.9% at 0% → 5.8% at 20%; every level ≥30% loses money. Pearson r = -0.814,Strong negative association; 30%+ discounts need control.
9,Which months have the highest sales?,"Order Date, Sales",Average monthly sales by calendar month,seasonality.png,"Top: Dec, Nov, Sep; weakest: Feb, Jan","Plan stock, staff and campaigns for Sep–Dec."
10,Is sales growth improving?,"Year, Sales, Profit",YoY % on yearly totals,monthly_sales.png,"Sales YoY: 8.0% (2023), 6.7% (2024). Profit YoY 2024: -6.6%","Sales growth is real, but 2024 profit fell → growth without profit."


---
## 3 · Deep dive A — Why did profit fall in 2024 while sales grew?

**Fact:** sales +6.7%, profit −6.6%. Let's find *where* the profit went by comparing 2024 with 2023.

In [5]:
prev, cur = LATEST - 1, LATEST
def change_by(col):
    p = df[df["Year"].isin([prev, cur])].pivot_table(index=col, columns="Year", values="Profit", aggfunc="sum")
    p["Change"] = p[cur] - p[prev]
    return p.sort_values("Change")

change_by("Region")

Year,2023,2024,Change
Region,,,
Central,"7,701.14",-744.27,"-8,445.41"
South,"14,913.56","13,725.75","-1,187.81"
East,"44,700.59","45,265.37",564.78
West,"50,755.90","51,994.91","1,239.01"


In [6]:
change_by("Sub-Category").head(5)

Year,2023,2024,Change
Sub-Category,,,
Tables,"-25,163.13","-32,026.23","-6,863.10"
Machines,"6,899.54","3,309.27","-3,590.27"
Storage,"11,016.09","8,762.77","-2,253.32"
Copiers,"20,326.99","18,403.30","-1,923.69"
Binders,"11,716.08","10,338.66","-1,377.42"


In [7]:
by_year = df.groupby("Year").apply(lambda g: pd.Series({
    "Effective Discount %": g["Discount Amount"].sum() / g["Gross Sales"].sum() * 100,
    "Profit on 30%+ discount lines": g.loc[g["Discount"] >= 0.3, "Profit"].sum(),
    "Tables sales": g.loc[g["Sub-Category"] == "Tables", "Sales"].sum(),
    "Tables profit": g.loc[g["Sub-Category"] == "Tables", "Profit"].sum(),
}), include_groups=False)
by_year

,Effective Discount %,Profit on 30%+ discount lines,Tables sales,Tables profit
Year,,,,
2021,15.07,"-51,616.30","103,533.88","-14,787.49"
2022,15.06,"-53,210.60","103,886.18","-19,544.26"
2023,15.59,"-63,061.77","85,019.43","-25,163.13"
2024,16.17,"-81,467.69","100,456.33","-32,026.23"


In [8]:
central_disc = (df[df["Region"] == "Central"].groupby("Year")[["Discount Amount", "Gross Sales"]].sum()
                .pipe(lambda t: t["Discount Amount"] / t["Gross Sales"] * 100))
central_disc.to_frame("Central effective discount %")

,Central effective discount %
Year,
2021,19.72
2022,21.47
2023,21.34
2024,23.68


**Facts:**
* Central's profit fell by ≈ $8.4K — the biggest regional drop — as its effective discount rose to ~23.7%.
* Tables sales *grew* in 2024, but Tables losses deepened by ≈ $6.9K: selling more of a loss-making product increases the loss.
* Company-wide effective discount rose from ~15.6% to ~16.2%; losses on 30%+ discount lines grew from ≈ −$63K to ≈ −$81K.

**Hypothesis:** part of 2024's sales growth was "bought" with deeper discounts, particularly in Central and on Tables. This fits every fact above, but the data cannot show *why* discounts were given (e.g. competitor pressure vs. sales targets). Interviews with Central sales managers would be the next step.

---
## 4 · Deep dive B — Tables: discount problem or pricing problem?

In [9]:
tables = df[df["Sub-Category"] == "Tables"]
tables.groupby("Discount Band").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).assign(
    **{"Margin %": lambda t: t["Profit"] / t["Sales"] * 100}).loc[["No discount", "1-20%", "21-40%", "Over 40%"]]

,Sales,Profit,Margin %
Discount Band,,,
No discount,"164,620.52","13,693.02",8.32
1-20%,"145,761.79","-16,728.06",-11.48
21-40%,"40,524.53","-18,024.48",-44.48
Over 40%,"41,988.98","-70,461.59",-167.81


**Fact:** Tables are only modestly profitable *without* discount (~8% margin), and already lose money at ordinary 10–20% discounts.
**Interpretation:** this is not only a discount problem — the **base price vs. cost** of Tables is too thin to support RetailCo's normal discount policy. Fixing discounts alone would reduce, not remove, the loss.

---
## 5 · What-if: capping discounts at 20%

**Question:** how much profit is at stake in the 30%+ discount lines?

**Method:** if a line sold at, say, 50% had instead been sold at 20%, its revenue rises by `Gross Sales × (Discount − 0.20)`. Cost doesn't change, so profit rises by the same amount.

⚠️ **This is an upper bound, not a forecast.** It assumes customers would have bought the *same quantity* at the higher price. In reality some would buy less or not at all. It tells management the *size of the opportunity*, not the guaranteed result.

In [10]:
deep = df[df["Discount"] >= 0.3]
uplift = (deep["Gross Sales"] * (deep["Discount"] - 0.20))
scenario = pd.DataFrame({
    "Lines affected": deep.groupby("Region").size(),
    "Max profit uplift": uplift.groupby(deep["Region"]).sum(),
})
scenario.loc["Total"] = scenario.sum()
print(f"Actual profit          : {money(df['Profit'].sum())} (margin {aggregate_margin(df['Sales'], df['Profit']):.1f}%)")
new_profit = df["Profit"].sum() + uplift.sum()
new_sales = df["Sales"].sum() + uplift.sum()
print(f"Upper-bound profit     : {money(new_profit)} (margin {new_profit / new_sales * 100:.1f}%)")
scenario

Actual profit          : $452K (margin 10.3%)
Upper-bound profit     : $716K (margin 15.4%)


,Lines affected,Max profit uplift
Region,,
Central,705.00,"102,178.76"
East,422.00,"48,015.61"
South,304.00,"42,814.23"
West,413.00,"71,258.06"
Total,"1,844.00","264,266.66"


Even if half of those customers walked away, the opportunity would still be large — which is why a **controlled test** (cap discounts in one Central sub-region first, compare with a similar region) is recommended before a company-wide rule.

---
## 6 · Recommendations

| # | Recommendation | Evidence (fact) | Owner | KPI to track |
|---|---|---|---|---|
| 1 | Require manager approval for discounts ≥ 30%; pilot first in Central | Every 30%+ level is loss-making; Central has 28% of lines at 30%+ | Sales Director, Central Regional Manager | Effective Discount %, Central margin |
| 2 | Review Tables & Bookcases pricing, supplier cost and minimum price | −$91.5K and −$30.4K; Tables lose money even at 10–20% discount | Category Manager, Procurement | Sub-category margin |
| 3 | Add profit/margin to sales incentives, not revenue alone | 2024: sales +6.7% while profit −6.6% | CFO, HR (compensation) | YoY Profit Growth vs YoY Sales Growth |
| 4 | Diagnose South's decline (market, competitors, sales coverage) before investing | South sales CAGR −12.2% while margin is healthy (10%) | Regional Manager South | South YoY sales, customer count |
| 5 | Protect and grow West/East; prepare Q4 capacity from August | West +14%/yr, East +12%/yr; Q4 ≈ 37% of sales | Sales & Operations | Monthly sales vs seasonal baseline |
| 6 | Key-account programme for the top 20% of customers, priced on margin | Top 20% of customers = 54% of sales; some top customers at ~1% margin | Account Management | Top-customer retention and margin |

**Expected impact:** the what-if gives an *upper bound* of ≈ $264K extra profit from discount control. A realistic target should be set after the Central pilot measures how volume reacts.